In [ ]:
import sys
import itertools
sys.path.append('../')
import random
from core import LADTransferTreeBoost, LSTransferTreeBoost
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb #baseline
from utils import *
from baselines import *
from friedman1 import *
import friedman_config as c

In [ ]:
#ablation study for transfertreeboost Gaussian errors, with gaussian source domain errors
ablation_transfer_normal_normal = pd.DataFrame(columns = ['seed', 'target_instances', 'd', 'method',
                                   'v', 'target_tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])


for seed in c.seed_list:
    for target_instances in c.target_instances_list:
    
        X_target_test, y_target_test = friedman1(n_samples=c.test_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
        X_target_val, y_target_val = friedman1(n_samples=c.val_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
        X_target_train, y_target_train = friedman1(n_samples=target_instances, add_noise = True, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #add noise to train set
        ones = np.ones((len(X_target_train), 1))
        X_target_train_with_dummy = np.hstack((X_target_train, ones))
        ones = np.ones((len(X_target_val), 1))
        X_target_val_with_dummy = np.hstack((X_target_val, ones))
        ones = np.ones((len(X_target_test), 1))
        X_target_test_with_dummy= np.hstack((X_target_test, ones))
        for d in c.d_list:
            X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'gaussian',
            
                                                        n_features=10, d=d, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)
            #Create an additional set with dummy variable for source/train for naive method!           
            zeros = np.zeros((len(X_source_train), 1))
            X_source_train_with_dummy = np.hstack((X_source_train.copy(), zeros))

            for config in c.param_grid_XGBoost:
                v, target_tree_size = config


                method = 'xgboost'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }
                        
                bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds_val = test_xgboost(X_target_val, bst)
                val_rmse = compute_rmse(preds_val, y_target_val)
                val_mae = compute_mae(preds_val, y_target_val)
                preds = test_xgboost(X_target_test, bst)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, v, target_tree_size, val_rmse, val_mae,
                                                                                              rmse,mae]
                
                ablation_transfer_normal_normal.to_csv(f'results/xgboost_ablation_friedman.csv')

                method = 'xgboost_naive_transfer_with_dummy'
                params = {
                    'objective': 'reg:squarederror',  # Regression with squared error
                    'max_depth': target_tree_size,                   # Maximum depth of a tree
                    'eta': v,                       # Learning rate
                    'eval_metric': 'rmse',           # RMSE as evaluation metric
                    }

                X_comb = np.concatenate((X_target_train_with_dummy, X_source_train_with_dummy)) 
                y_comb = np.concatenate((y_target_train, y_source_train))       
                bst = train_xgboost(X_comb, y_comb, X_target_val, y_target_val, boosting_rounds=1000, params=params)
                preds_val = test_xgboost(X_target_val_with_dummy, bst)
                val_rmse = compute_rmse(preds_val, y_target_val)
                val_mae = compute_mae(preds_val, y_target_val)
                preds = test_xgboost(X_target_test_with_dummy, bst)
                rmse = compute_rmse(preds, y_target_test)
                mae = compute_mae(preds, y_target_test)
                ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, v, target_tree_size, val_rmse, val_mae,
                                                                                              rmse,mae]
                
                ablation_transfer_normal_normal.to_csv(f'results/xgboost_ablation_friedman.csv')
                                
            

